# 05 - ModernBERT Fine-Tuning (v2 Corrected Labels)

Full encoder fine-tuning of ModernBERT-base (150M params) on Sonnet-corrected labels.
This establishes the quality ceiling for a transformer model with correct labels.

**v1 baseline:** 61.3% top-1 on noisy Kaggle labels
**v2 MLP baseline:** 84.9% top-1 with corrected labels + distillation
**Expected:** ModernBERT on corrected labels should exceed 90% top-1

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score

PROJECT_DIR = Path('.').resolve().parent.parent
DATA_DIR = PROJECT_DIR / 'data' / 'corrected'
MODEL_DIR = PROJECT_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

print(f'PyTorch: {torch.__version__}')
print(f'MPS available: {torch.backends.mps.is_available()}')
print(f'Device: mps')

In [ ]:
# Load corrected data
train_df = pd.read_parquet(DATA_DIR / 'train.parquet')
val_df = pd.read_parquet(DATA_DIR / 'val.parquet')
test_df = pd.read_parquet(DATA_DIR / 'test.parquet')

print(f'Train: {len(train_df):,}')
print(f'Val:   {len(val_df):,}')
print(f'Test:  {len(test_df):,}')
print(f'Classes: {train_df["tier1"].nunique()}')

In [ ]:
# Encode labels
le = LabelEncoder()
le.fit(sorted(train_df['tier1'].unique()))

train_df['label'] = le.transform(train_df['tier1'])
val_df['label'] = le.transform(val_df['tier1'])
test_df['label'] = le.transform(test_df['tier1'])

num_labels = len(le.classes_)
print(f'Labels: {num_labels} classes')
print(f'Classes: {le.classes_[:5]}...')

In [ ]:
# Prepare text -- use enriched 'text' column (domain | title | keywords)
def prepare_texts(df):
    texts = df['text'].fillna(df['domain_clean']).tolist()
    return [t[:256] for t in texts]

train_texts = prepare_texts(train_df)
val_texts = prepare_texts(val_df)
test_texts = prepare_texts(test_df)

train_labels = train_df['label'].values
val_labels = val_df['label'].values
test_labels = test_df['label'].values

print(f'Text samples:')
for t in train_texts[:3]:
    print(f'  {t[:100]}...')

In [ ]:
# Disable SSL for HuggingFace
import httpx
from huggingface_hub.utils._http import set_client_factory, hf_request_event_hook

def no_ssl_client_factory():
    return httpx.Client(
        event_hooks={'request': [hf_request_event_hook]},
        follow_redirects=True,
        timeout=None,
        verify=False,
    )

set_client_factory(no_ssl_client_factory)
print('SSL disabled for HuggingFace')

In [ ]:
# Load ModernBERT tokenizer and model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'answerdotai/ModernBERT-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}')
print(f'Parameters: {total_params:,}')

In [ ]:
# Tokenize
from torch.utils.data import Dataset

class DomainDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts, truncation=True, padding='max_length',
                                   max_length=max_length, return_tensors='pt')
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = DomainDataset(train_texts, train_labels, tokenizer)
val_dataset = DomainDataset(val_texts, val_labels, tokenizer)
test_dataset = DomainDataset(test_texts, test_labels, tokenizer)

print(f'Train dataset: {len(train_dataset):,}')
print(f'Val dataset: {len(val_dataset):,}')
print(f'Test dataset: {len(test_dataset):,}')

In [ ]:
# Compute class weights for imbalanced data
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.arange(num_labels), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print(f'Class weights computed (range: {class_weights.min():.2f} to {class_weights.max():.2f})')

In [ ]:
# Training with HuggingFace Trainer
from transformers import TrainingArguments, Trainer
import time

training_args = TrainingArguments(
    output_dir=str(MODEL_DIR / 'modernbert_v2_checkpoints'),
    num_train_epochs=3,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=2,
    fp16=False,
    dataloader_num_workers=0,
    report_to='none',
)

import torch.nn as nn

class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    # Top-3
    top3 = top_k_accuracy_score(labels, logits, k=3, labels=np.arange(num_labels))
    return {'accuracy': acc, 'top3_accuracy': top3}


trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print(f'Training config:')
print(f'  Epochs: 3')
print(f'  Batch size: 64')
print(f'  LR: 2e-5')
print(f'  Warmup: 10%')
print(f'  Class-weighted loss: yes')
print(f'\nStarting training...')

start_time = time.time()
trainer.train()
elapsed = time.time() - start_time
print(f'\nTraining complete in {elapsed/60:.1f} minutes')

In [ ]:
# Evaluate on validation set
val_results = trainer.evaluate(val_dataset)
print(f'VALIDATION RESULTS')
print(f'=' * 50)
print(f'  Top-1 Accuracy: {val_results["eval_accuracy"]:.1%}')
print(f'  Top-3 Accuracy: {val_results["eval_top3_accuracy"]:.1%}')
print(f'\nComparison:')
print(f'  v1 ModernBERT (noisy labels):    61.3% top-1')
print(f'  v2 MLP (corrected + distilled):  84.9% top-1')
print(f'  v2 ModernBERT (corrected labels): {val_results["eval_accuracy"]:.1%} top-1')

In [ ]:
# Per-category report
val_preds = trainer.predict(val_dataset)
val_pred_labels = np.argmax(val_preds.predictions, axis=1)

print(f'PER-CATEGORY CLASSIFICATION REPORT')
print(f'=' * 80)
print(classification_report(val_labels, val_pred_labels, target_names=le.classes_, digits=3))

In [ ]:
# Save best model
import json

save_dir = MODEL_DIR / 'modernbert_v2_best'
save_dir.mkdir(exist_ok=True)
trainer.save_model(str(save_dir))
tokenizer.save_pretrained(str(save_dir))

# Save metadata
meta = {
    'model': MODEL_NAME,
    'val_top1_accuracy': float(val_results['eval_accuracy']),
    'val_top3_accuracy': float(val_results['eval_top3_accuracy']),
    'training_time_min': elapsed / 60,
    'epochs': 3,
    'batch_size': 64,
    'lr': 2e-5,
    'class_weighted': True,
    'text_format': 'domain | title | sonnet_keywords',
    'labels': le.classes_.tolist(),
    'v1_comparison': {'v1_modernbert_top1': 0.613, 'v2_mlp_top1': 0.849},
}
with open(save_dir / 'meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

model_size = sum(f.stat().st_size for f in save_dir.rglob('*') if f.is_file()) / 1e6
print(f'Model saved to {save_dir} ({model_size:.0f} MB)')

In [ ]:
# Final comparison table
print(f'\n{"="*70}')
print(f'FULL MODEL COMPARISON')
print(f'{"="*70}')
print(f'{"Model":<45} | {"Top-1":>7} | {"Top-3":>7} | {"Params":>8}')
print(f'{"-"*70}')
print(f'{"v1 MLP (noisy labels, 8% teacher)":<45} | {"45.1%":>7} | {"68.0%":>7} | {"135K":>8}')
print(f'{"v1 ModernBERT (noisy labels)":<45} | {"61.3%":>7} | {"80.0%":>7} | {"150M":>8}')
print(f'{"v2 MLP (corrected, 42% teacher)":<45} | {"84.9%":>7} | {"98.3%":>7} | {"337K":>8}')
v2_top1 = val_results['eval_accuracy']
v2_top3 = val_results['eval_top3_accuracy']
print(f'{"v2 ModernBERT (corrected labels)":<45} | {v2_top1*100:.1f}% | {v2_top3*100:.1f}% | {"150M":>8}')
print(f'{"-"*70}')

## Interpretation

Results will be analyzed after execution with actual numbers.